# U3-04 · Algoritmos desde cero y scikit-learn

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 3, apartados 3.7 a 3.9*

Reproduce los resultados de los apartados 3.8 y 3.9 del libro.

Ejecuta cada celda con **Mayús + Intro**. Cambia lo que quieras y vuelve a ejecutar: no se puede romper nada.

## 1. Regresión lineal con las ecuaciones normales

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

carga = np.array([20, 30, 40, 50, 60, 70, 80, 90, 35, 55, 75, 85])
consumo = np.array([5.8, 7.5, 8.3, 9.7, 10.6, 12.2,
                    13.5, 15.7, 8.3, 10.1, 12.5, 14.4])
X = np.column_stack([np.ones(len(carga)), carga])
beta = np.linalg.solve(X.T @ X, X.T @ consumo)
print(beta.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.scatter(carga, consumo, color="#1F3A40", label="medidas")
x = np.linspace(20, 90, 50)
ax.plot(x, beta[0] + beta[1] * x, color="#1C8C96", label="recta ajustada")
ax.set_xlabel("Carga (%)"); ax.set_ylabel("Consumo (kWh)"); ax.legend()
plt.show()

## 2. k vecinos más próximos desde cero

In [ ]:
def knn_predecir(X_ent, y_ent, X_nuevos, k=5):
    predicciones = []
    for x in X_nuevos:
        dist = np.sqrt(((X_ent - x) ** 2).sum(axis=1))  # distancia euclídea
        vecinos = np.argsort(dist)[:k]                 # los k más cercanos
        clases, votos = np.unique(y_ent[vecinos], return_counts=True)
        predicciones.append(clases[np.argmax(votos)])  # clase mayoritaria
    return np.array(predicciones)

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

X, y = load_wine(return_X_y=True)
print(X.shape, np.bincount(y))
X_ent, X_pru, y_ent, y_pru = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
esc = StandardScaler().fit(X_ent)            # solo con los datos de entrenamiento
Xe, Xp = esc.transform(X_ent), esc.transform(X_pru)

p_propio = knn_predecir(Xe, y_ent, Xp, k=5)
p_sklearn = KNeighborsClassifier(n_neighbors=5).fit(Xe, y_ent).predict(Xp)
print("Exactitud propia:  ", (p_propio == y_pru).mean().round(3))
print("Exactitud sklearn: ", (p_sklearn == y_pru).mean().round(3))
print("Mismas predicciones:", (p_propio == p_sklearn).all())

In [ ]:
# ¿Y sin estandarizar?
p_crudo = KNeighborsClassifier(n_neighbors=5).fit(X_ent, y_ent).predict(X_pru)
print("Exactitud sin estandarizar:", (p_crudo == y_pru).mean().round(3))

## 3. Pipeline, validación cruzada y búsqueda de k

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV

modelo = make_pipeline(StandardScaler(), KNeighborsClassifier())
print("Validación cruzada (k=5):", cross_val_score(modelo, X_ent, y_ent, cv=5).mean().round(3))

rejilla = {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21]}
busqueda = GridSearchCV(modelo, rejilla, cv=5).fit(X_ent, y_ent)
print(busqueda.best_params_, round(busqueda.best_score_, 3))
print("Exactitud en prueba:", round(busqueda.score(X_pru, y_pru), 3))

In [ ]:
import pandas as pd
res = pd.DataFrame(busqueda.cv_results_)
plt.figure(figsize=(5, 3))
plt.plot(res["param_kneighborsclassifier__n_neighbors"].astype(int), res["mean_test_score"], "o-", color="#1C8C96")
plt.xlabel("k (número de vecinos)"); plt.ylabel("Exactitud media en validación")
plt.show()

**Ejercicio.** Sustituye `KNeighborsClassifier()` por `LogisticRegression(max_iter=1000)` (de `sklearn.linear_model`) y quita la búsqueda de *k*. ¿Cuántas líneas has tenido que cambiar?

> Cover, T. M. y Hart, P. E. (1967). Nearest neighbor pattern classification. *IEEE Trans. Information Theory*, 13(1), 21–27. https://doi.org/10.1109/TIT.1967.1053964  
> Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.